# gsplat Inference Example

This notebook demonstrates how to perform a simple forward pass using the `gsplat` rasterization API. We will initialize dummy 3D Gaussians and camera parameters, then render them into a 2D image.

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from gsplat.rendering import rasterization

## 1. Initialize Gaussians

We create a small batch of 1000 random 3D Gaussians. Each Gaussian has:
- **means**: 3D center $(x, y, z)$
- **quats**: Rotation quaternions
- **scales**: 3D scaling factors
- **opacities**: Alpha blending weight
- **colors**: RGB values

In [ ]:
# Device configuration (use CUDA if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

torch.manual_seed(42)
num_gaussians = 1000

# Randomly place means within a small volume
means = (torch.rand((num_gaussians, 3), device=device) - 0.5) * 5.0
means[..., 2] -= 5.0 # Move them away from the camera along the Z axis

# Random quaternions
quats = F.normalize(torch.randn((num_gaussians, 4), device=device), dim=-1)

# Scales (exponentiated so they are positive)
scales = torch.rand((num_gaussians, 3), device=device) * 0.1

# Opacities between 0.5 and 1.0
opacities = torch.rand((num_gaussians,), device=device) * 0.5 + 0.5

# Random RGB colors
colors = torch.rand((num_gaussians, 3), device=device)

## 2. Initialize Camera Parameters

We need to define the camera extrinsics (`viewmats`) and intrinsics (`Ks`), as well as the resolution.

In [ ]:
width, height = 512, 512
focal_length = 500.0

# Intrinsics matrix (K)
K = torch.tensor([
    [focal_length, 0.0, width / 2.0],
    [0.0, focal_length, height / 2.0],
    [0.0, 0.0, 1.0]
], device=device)

# View matrix (Identity - looking down -Z axis)
viewmat = torch.eye(4, device=device)

# The API expects batch and camera dimensions: [..., C, 4, 4]
# So we add a camera dimension of 1.
viewmats = viewmat.unsqueeze(0) # [1, 4, 4]
Ks = K.unsqueeze(0) # [1, 3, 3]

## 3. Rasterization

Run the forward pass to get the rendered image and alpha mask.

In [ ]:
render_colors, render_alphas, meta = rasterization(
    means=means,
    quats=quats,
    scales=scales,
    opacities=opacities,
    colors=colors,
    viewmats=viewmats,
    Ks=Ks,
    width=width,
    height=height,
    packed=False,  # Use un-packed mode for simplicity in small examples
)

print("Rendered shape:", render_colors.shape)

# Extract the single image from the [C, H, W, 3] tensor
image = render_colors[0].detach().cpu().numpy()
alpha = render_alphas[0].detach().cpu().numpy()

## 4. Visualize the Output

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 5))

axs[0].imshow(np.clip(image, 0.0, 1.0))
axs[0].set_title("Rendered Colors")
axs[0].axis('off')

axs[1].imshow(alpha.squeeze(), cmap='gray', vmin=0, vmax=1)
axs[1].set_title("Alpha Mask")
axs[1].axis('off')

plt.show()